[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S10/S10_04_reto_vale_la_pena.ipynb)

# S10 · Reto PBL — ¿Vale la pena la red?

**Tiempo estimado:** 1 h 30 min de guía en clase (el reto completo son 4 h de PBL) · **Sprint 10**

**Escena del cliente.** La dirección de TurisData Canarias pregunta: *"Todo el mundo habla de redes neuronales.
¿Debemos cambiar nuestro modelo de cancelaciones por una red?"* Tu tarea no es "hacer que la red gane", sino
**responder con evidencias**: comparar un MLP con un modelo de árbol/boosting, medir el coste y diagnosticar
el sobreajuste con curvas.

**Este notebook** te da el esqueleto y las piezas de comparación para las cancelaciones. En tu entregable
(notebook + informe comparativo) puedes elegir la variante de precios, otra arquitectura, etc.

**Aviso sobre el "campeón del Sprint 8".** Para que el notebook funcione solo, aquí entrenas un
`HistGradientBoostingClassifier` (boosting de árboles). Si tu equipo tiene su propio campeón del Sprint 8
(XGBoost, LightGBM, Random Forest…), úsalo en su lugar y compara con él.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os, time
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score, mean_absolute_error

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)

## 1. Datos y partición (la misma para todos los modelos)
Comparar modelos con particiones distintas no es justo. Usamos **una** partición: 64 % entrenamiento,
16 % validación (decisiones y curvas) y 20 % prueba (veredicto final, una sola vez).

In [ ]:
df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
num = ["antelacion_dias", "noches", "adultos", "ninos", "peticiones_especiales", "cliente_repetidor",
       "cancelaciones_previas", "mes_llegada", "precio_noche"]
cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[num + cat], df["cancelada"].to_numpy()
X_res, X_pru, y_res, y_pru = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
X_ent, X_val, y_ent, y_val = train_test_split(X_res, y_res, test_size=0.2, random_state=SEMILLA, stratify=y_res)
pre = ColumnTransformer([("n", StandardScaler(), num), ("c", OneHotEncoder(handle_unknown="ignore"), cat)])
A_ent = pre.fit_transform(X_ent).astype("float32")
A_val = pre.transform(X_val).astype("float32")
A_pru = pre.transform(X_pru).astype("float32")
print(A_ent.shape, A_val.shape, A_pru.shape)

## 2. Una función para evaluar a todos por igual
**Ejercicio 1.** Escribe `evaluar(y_real, prob, umbral=0.5)` que devuelva un diccionario con cuatro claves:
`"AUC"` (`roc_auc_score`), `"AP"` (`average_precision_score`, área bajo la curva precisión-recall),
`"F1"` y `"precisión"` (`accuracy_score`) — las dos últimas con las clases obtenidas al aplicar el `umbral` a `prob`.

In [ ]:
# TODO: Convierte prob a clases con (prob >= umbral).astype(int) y llama a las métricas de sklearn
...

_r = evaluar(np.array([0, 0, 1, 1]), np.array([0.1, 0.6, 0.4, 0.9]))
assert set(_r) == {"AUC", "AP", "F1", "precisión"}, "El diccionario debe tener las claves AUC, AP, F1 y precisión"
assert abs(_r["AUC"] - 0.75) < 1e-9 and abs(_r["precisión"] - 0.5) < 1e-9, "Revisa: AUC esperado 0.75 y precisión 0.5 en la prueba mínima"

## 3. La referencia: boosting de árboles
Se entrena con los datos originales (los árboles no necesitan escalar), pero usamos las mismas filas.
Medimos también el **tiempo**: forma parte del coste de "la red".

In [ ]:
t0 = time.time()
boost = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=4, random_state=SEMILLA)
boost.fit(A_ent, y_ent)
t_boost = time.time() - t0
prob_boost_val, prob_boost_pru = boost.predict_proba(A_val)[:, 1], boost.predict_proba(A_pru)[:, 1]
print(f"Boosting entrenado en {t_boost:.1f} s · AUC validación = {roc_auc_score(y_val, prob_boost_val):.3f}")

## 4. El MLP, con las buenas prácticas del Sprint 10
Pequeño, con L2, dropout y parada temprana. Para no depender de la suerte de una semilla, lo entrenamos
**con 3 semillas** y damos la media y la variación.

In [ ]:
def construir_mlp(n_entradas, capas=(64, 32), l2=0.003, dropout=0.3, tasa=1e-3):
    m = keras.Sequential([keras.layers.Input(shape=(n_entradas,))])
    for u in capas:
        m.add(keras.layers.Dense(u, activation="relu", kernel_regularizer=keras.regularizers.l2(l2)))
        m.add(keras.layers.Dropout(dropout))
    m.add(keras.layers.Dense(1, activation="sigmoid"))
    m.compile(optimizer=keras.optimizers.Adam(tasa), loss="binary_crossentropy", metrics=[keras.metrics.AUC(name="auc")])
    return m

modelos_mlp, historiales, tiempos = [], [], []
for semilla in (1, 2, 3):
    keras.utils.set_random_seed(semilla)
    t0 = time.time()
    m = construir_mlp(A_ent.shape[1])
    h = m.fit(A_ent, y_ent, validation_data=(A_val, y_val), epochs=40, batch_size=64, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)])
    tiempos.append(time.time() - t0)
    modelos_mlp.append(m); historiales.append(h)
    print(f"semilla {semilla}: {len(h.history['loss'])} épocas, AUC validación = {max(h.history['val_auc']):.3f} (máx.)")

**Ejercicio 2.** Calcula en `aucs_mlp_val` una lista con el AUC de cada modelo de `modelos_mlp` sobre
la **validación** (usa `roc_auc_score(y_val, modelo.predict(A_val, verbose=0).ravel())`), y en
`auc_mlp_medio` su media.

In [ ]:
# TODO: Una lista por comprensión sobre modelos_mlp y np.mean
...

assert len(aucs_mlp_val) == 3, "Debe haber un AUC por cada uno de los 3 modelos"
assert 0.6 < auc_mlp_medio < 0.85, "El AUC medio debería rondar 0.7–0.75"
print("AUC de validación por semilla:", np.round(aucs_mlp_val, 3), "· media", round(auc_mlp_medio, 3))

## 5. Diagnóstico con curvas de aprendizaje
Con `EarlyStopping` la red se detiene a tiempo; aun así conviene mirar las curvas: ¿cuántas épocas útiles?
¿cuánta brecha entre entrenamiento y validación?

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for eje, h, s in zip(ejes, historiales, (1, 2, 3)):
    eje.plot(h.history["loss"], label="entrenamiento"); eje.plot(h.history["val_loss"], label="validación")
    eje.set_title(f"MLP · semilla {s}"); eje.set_xlabel("época"); eje.set_ylabel("pérdida (incluye L2)"); eje.legend()
plt.tight_layout(); plt.show()

**Ejercicio 3.** Escribe `diagnostico(historial, margen=0.03)` que reciba el objeto `History` de Keras y devuelva
el texto `"sobreajuste"` si la pérdida de validación **final** supera a su **mínimo** en más de `margen`
**o** si la pérdida de entrenamiento final es menor que la de validación final en más de `0.15`;
si no, devuelve `"sin sobreajuste claro"`.

In [ ]:
# TODO: Compara val_loss[-1] con min(val_loss) + margen, y la brecha val_loss[-1] - loss[-1] con 0.15
...

class _H:  # historial de juguete para probar la función
    def __init__(self, loss, val): self.history = {"loss": loss, "val_loss": val}
assert diagnostico(_H([0.6, 0.3, 0.1], [0.6, 0.65, 0.9])) == "sobreajuste", "Validación creciente y brecha grande = sobreajuste"
assert diagnostico(_H([0.7, 0.65, 0.62], [0.7, 0.66, 0.64])) == "sin sobreajuste claro", "Curvas paralelas y juntas: sin sobreajuste"
for s, h in zip((1, 2, 3), historiales):
    print(f"MLP semilla {s}:", diagnostico(h))

## 6. El veredicto en el conjunto de prueba
Ahora sí se usa la prueba: una sola vez. Para el MLP tomamos la **media de las 3 semillas** de la probabilidad
(un pequeño *ensemble*).

In [ ]:
prob_mlp_pru = np.mean([m.predict(A_pru, verbose=0).ravel() for m in modelos_mlp], axis=0)
tabla = pd.DataFrame({"Boosting de árboles": evaluar(y_pru, prob_boost_pru),
                      "MLP (media de 3 semillas)": evaluar(y_pru, prob_mlp_pru)}).T
tabla["entrenamiento (s)"] = [t_boost, np.mean(tiempos)]
tabla["parámetros / árboles"] = ["200 iteraciones × árboles de profundidad ≤ 4", f"{modelos_mlp[0].count_params():,} pesos"]
tabla.round(3)

### ¿La diferencia es real o es ruido?
Con 1200 reservas de prueba, un AUC de 0.74 frente a 0.73 puede ser pura casualidad. Estimamos la incertidumbre
de la **diferencia de AUC** remuestreando la prueba (bootstrap): si el intervalo incluye el 0, no podemos afirmar que
un modelo sea mejor.

In [ ]:
rng = np.random.default_rng(SEMILLA)
difs = []
for _ in range(300):
    idx = rng.integers(0, len(y_pru), len(y_pru))
    if len(np.unique(y_pru[idx])) < 2:
        continue
    difs.append(roc_auc_score(y_pru[idx], prob_mlp_pru[idx]) - roc_auc_score(y_pru[idx], prob_boost_pru[idx]))
inf, sup = np.percentile(difs, [2.5, 97.5])
print(f"Diferencia de AUC (MLP − boosting): {np.mean(difs):+.3f} · intervalo 95 %: [{inf:+.3f}, {sup:+.3f}]")
plt.hist(difs, bins=25, color="steelblue"); plt.axvline(0, color="red", ls="--", label="sin diferencia")
plt.xlabel("AUC MLP − AUC boosting"); plt.ylabel("frecuencia"); plt.title("Incertidumbre de la comparación (bootstrap)")
plt.legend(); plt.show()

**Ejercicio 4.** Guarda en `hay_diferencia` un booleano: `True` si el intervalo `[inf, sup]` **no** contiene el 0
(es decir, si `inf > 0` o `sup < 0`). Guarda en `mejor_modelo` el texto `"MLP"`, `"Boosting"` o `"Empate"` según
el resultado (`"Empate"` si no hay diferencia clara).

In [ ]:
# TODO: hay_diferencia = inf > 0 or sup < 0; si la hay, gana el MLP si la media es positiva
...

assert isinstance(hay_diferencia, bool), "hay_diferencia debe ser True/False"
assert mejor_modelo in ("MLP", "Boosting", "Empate"), "mejor_modelo debe ser 'MLP', 'Boosting' o 'Empate'"
assert (mejor_modelo == "Empate") == (not hay_diferencia), "Si no hay diferencia clara debe ser 'Empate' (y viceversa)"
print("Veredicto estadístico:", mejor_modelo)

## 7. Variante de regresión (opcional, precio por noche)
El mismo razonamiento para `precio_noche`. Recuerda quitar `precio_noche` de las entradas.

In [ ]:
num_r = [c for c in num if c != "precio_noche"]
Xr = df[num_r + cat]; yr = df["precio_noche"].to_numpy()
Xr_res, Xr_pru, yr_res, yr_pru = train_test_split(Xr, yr, test_size=0.2, random_state=SEMILLA)
pre_r = ColumnTransformer([("n", StandardScaler(), num_r), ("c", OneHotEncoder(handle_unknown="ignore"), cat)])
Ar_res = pre_r.fit_transform(Xr_res).astype("float32"); Ar_pru = pre_r.transform(Xr_pru).astype("float32")
mu, sd = yr_res.mean(), yr_res.std()

boost_r = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=4, random_state=SEMILLA).fit(Ar_res, yr_res)
ridge_r = Ridge().fit(Ar_res, yr_res)
keras.utils.set_random_seed(SEMILLA)
mlp_r = keras.Sequential([keras.layers.Input(shape=(Ar_res.shape[1],)), keras.layers.Dense(32, activation="relu"),
                          keras.layers.Dense(16, activation="relu"), keras.layers.Dense(1)])
mlp_r.compile("adam", "mse")
mlp_r.fit(Ar_res, ((yr_res - mu) / sd).astype("float32"), validation_split=0.2, epochs=40, batch_size=64, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])
pd.DataFrame({"MAE (€)": {
    "Boosting": mean_absolute_error(yr_pru, boost_r.predict(Ar_pru)),
    "Ridge (lineal)": mean_absolute_error(yr_pru, ridge_r.predict(Ar_pru)),
    "MLP": mean_absolute_error(yr_pru, mlp_r.predict(Ar_pru, verbose=0).ravel() * sd + mu)}}).round(2)

## 8. Tu informe: la recomendación para TurisData
Rellena (en el informe comparativo, no aquí) esta estructura de una página:
1. **Pregunta y respuesta en una frase** ("Recomendamos mantener / cambiar a … porque …").
2. **Tabla** de resultados en prueba con incertidumbre (AUC, AP, F1) y **coste** (tiempo, complejidad, explicabilidad).
3. **Curvas de aprendizaje** y diagnóstico de sobreajuste, con lo que hiciste para controlarlo.
4. **Límites y riesgos** (datos sintéticos, umbral de decisión, qué pasa con reservas nuevas, sesgos por país o canal).
5. **Siguiente paso** recomendado.

## 9. Preguntas de reflexión
1. Si la red y el boosting empatan, ¿cuál recomendarías a un hotel con un solo técnico? ¿Por qué?
2. ¿En qué tipo de datos esperarías que la red sí ganara claramente (imágenes, texto, series)? ¿Por qué no aquí?
3. ¿Qué coste tiene equivocarse cancelando "de más" en el overbooking frente a "de menos"? ¿Cómo movería eso el umbral?

## 10. Resumen: qué has aprendido
- "Vale la pena" se responde con **métrica + incertidumbre + coste**, no con la moda tecnológica.
- En datos tabulares pequeños, el boosting suele igualar o superar al MLP con menos esfuerzo (aquí: empate en cancelaciones y victoria clara del boosting en precio).
- Las curvas de aprendizaje y la parada temprana evitan que la red "presuma" de un entrenamiento memorizado.
- Se compara con la misma partición, varias semillas y un intervalo de confianza.